In [1]:
!unzip /content/DL_Assignment1_Dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: Dataset/Dataset/annotations/691_aro.npy  
  inflating: Dataset/Dataset/annotations/691_exp.npy  
  inflating: Dataset/Dataset/annotations/691_lnd.npy  
  inflating: Dataset/Dataset/annotations/691_val.npy  
  inflating: Dataset/Dataset/annotations/692_aro.npy  
  inflating: Dataset/Dataset/annotations/692_exp.npy  
  inflating: Dataset/Dataset/annotations/692_lnd.npy  
  inflating: Dataset/Dataset/annotations/692_val.npy  
  inflating: Dataset/Dataset/annotations/693_aro.npy  
  inflating: Dataset/Dataset/annotations/693_exp.npy  
  inflating: Dataset/Dataset/annotations/693_lnd.npy  
  inflating: Dataset/Dataset/annotations/693_val.npy  
  inflating: Dataset/Dataset/annotations/694_aro.npy  
  inflating: Dataset/Dataset/annotations/694_exp.npy  
  inflating: Dataset/Dataset/annotations/694_lnd.npy  
  inflating: Dataset/Dataset/annotations/694_val.npy  
  inflating: Dataset/Dataset/annotations/695_aro.npy  
  inflating: D

In [2]:
# Check GPU
import torch, platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Set your project root in Colab (where you upload all files: main.py, models, etc.)
PROJECT_ROOT = "/content/"   # change only if you uploaded elsewhere
DATA_ROOT = f"{PROJECT_ROOT}/Dataset"  # must contain images/ and annotations/

# Create folders if needed (you will upload files into /content/DL_a2)
os.makedirs(PROJECT_ROOT, exist_ok=True)
print("Project root:", PROJECT_ROOT)

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.6.97+-x86_64-with-glibc2.35
CUDA available: True
GPU: Tesla T4
Project root: /content/DL_a2


In [3]:
# Install requirements
%pip install -q -r /content/DL_a2/requirements.txt

# On Colab, torch/torchvision are usually installed already with CUDA.
# If not present or mismatched, uncomment the following to install matching versions:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

  Preparing metadata (setup.py) ... done


In [4]:
# Runs your single-file pipeline
%cd /content/DL_a2
!python main.py --phase data_exploration --data_root "{DATA_ROOT}"

/content/DL_a2
Dataset root: /content/DL_a2/Dataset
Images found: 3999
NPY files in annotations/: 15996 (show 5)
- 4004_val.npy: shape=() dtype=<U9
- 3039_exp.npy: shape=() dtype=<U1
- 698_aro.npy: shape=() dtype=<U9
- 2950_lnd.npy: shape=(136,) dtype=float64
- 899_val.npy: shape=() dtype=<U9


In [5]:
# Training both ResNet-50 and EfficientNet-B0
# Adjust epochs as needed. AMP will be used automatically if CUDA is available.
EPOCHS = 50
BATCH = 32
NUM_WORKERS = 2  # On Colab GPU, you can set >0

!python main.py --phase train_models --data_root "{DATA_ROOT}" --epochs {EPOCHS} --batch_size {BATCH} --lr 1e-3 --num_workers {NUM_WORKERS}

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth
100% 20.4M/20.4M [00:00<00:00, 98.2MB/s]
Loaded pretrained weights for efficientnet-b0
/content/DL_a2/main.py:431: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
/content/DL_a2/main.py:344: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
[resnet50] Epoch 1/50 train={'loss': 2.325180644989014, 'cls': 2.1862358593940736, 'reg': 0.27582929611206053, 'acc': 0.11910282254219055, 'mae': 0.4098850876092911} val={'loss': 2.2577522563934327, 'cls': 2.0817853355407716, 'reg': 0.17596690654754638, 'acc': 0.1175, 'mae': 0.349070422649383

In [13]:
# Use the known working paths
resnet_ckpt = "/content/DL_a2/checkpoints_both/20250927_163355/resnet50/best.pth"
effnet_ckpt = "/content/DL_a2/checkpoints_both/20250927_165512/efficientnet_b0/best.pth"

# But first check if they exist
import os
print(f"ResNet checkpoint exists: {os.path.exists(resnet_ckpt)}")
print(f"EfficientNet checkpoint exists: {os.path.exists(effnet_ckpt)}")

if os.path.exists(resnet_ckpt) and os.path.exists(effnet_ckpt):
    !python main.py --phase evaluate_models --data_root "{DATA_ROOT}" \
        --resnet_ckpt "{resnet_ckpt}" --effnet_ckpt "{effnet_ckpt}" \
        --batch_size 64 --num_workers {NUM_WORKERS} --eval_out eval_outputs

ResNet checkpoint exists: True
EfficientNet checkpoint exists: True
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [14]:
# Generates a short markdown summary linking parameter counts and available metrics
!python main.py --phase generate_report --data_root "{DATA_ROOT}" --resnet_ckpt "{resnet_ckpt}" --effnet_ckpt "{effnet_ckpt}" --eval_out eval_outputs --report_out report_outputs

print("Check eval_outputs/ and report_outputs/ for artifacts.")

Report generated: report_outputs/report_20250927_173027.md
Check eval_outputs/ and report_outputs/ for artifacts.


In [15]:
!zip -r checkpoints_both.zip checkpoints_both/

  adding: checkpoints_both/ (stored 0%)
  adding: checkpoints_both/20250927_165512/ (stored 0%)
  adding: checkpoints_both/20250927_165512/efficientnet_b0/ (stored 0%)
  adding: checkpoints_both/20250927_165512/efficientnet_b0/best.pth (deflated 8%)
  adding: checkpoints_both/20250927_165512/efficientnet_b0/last.pth (deflated 8%)
  adding: checkpoints_both/20250927_165512/efficientnet_b0/history.json (deflated 73%)
  adding: checkpoints_both/20250927_163355/ (stored 0%)
  adding: checkpoints_both/20250927_163355/resnet50/ (stored 0%)
  adding: checkpoints_both/20250927_163355/resnet50/best.pth (deflated 8%)
  adding: checkpoints_both/20250927_163355/resnet50/last.pth (deflated 8%)
  adding: checkpoints_both/20250927_163355/resnet50/history.json (deflated 74%)


In [16]:
!zip -r eval_outputs.zip eval_outputs/

  adding: eval_outputs/ (stored 0%)
  adding: eval_outputs/20250927_172841/ (stored 0%)
  adding: eval_outputs/20250927_172841/scatter_val_efficientnet_b0.png (deflated 8%)
  adding: eval_outputs/20250927_172841/pr_resnet50.png (deflated 8%)
  adding: eval_outputs/20250927_172841/scatter_val_resnet50.png (deflated 7%)
  adding: eval_outputs/20250927_172841/cm_efficientnet_b0.png (deflated 13%)
  adding: eval_outputs/20250927_172841/roc_efficientnet_b0.png (deflated 13%)
  adding: eval_outputs/20250927_172841/summary.json (deflated 77%)
  adding: eval_outputs/20250927_172841/scatter_aro_resnet50.png (deflated 10%)
  adding: eval_outputs/20250927_172841/scatter_aro_efficientnet_b0.png (deflated 7%)
  adding: eval_outputs/20250927_172841/metrics_resnet50.json (deflated 70%)
  adding: eval_outputs/20250927_172841/pr_efficientnet_b0.png (deflated 6%)
  adding: eval_outputs/20250927_172841/cm_resnet50.png (deflated 14%)
  adding: eval_outputs/20250927_172841/roc_resnet50.png (deflated 12%)
 

In [17]:
!zip -r report_outputs.zip report_outputs/

  adding: report_outputs/ (stored 0%)
  adding: report_outputs/report_20250927_173027.md (deflated 35%)
